# UR10e 3D Fruit Ninja Robot Simulation

This notebook simulates a UR10e robot arm with "fruit" balls that can be touched by the end effector.

- **Cell 1**: Setup - imports, classes, and model creation
- **Cell 2**: User Control - set joint torques for each timestep. ONLY EDIT THIS
- **Cell 3**: Simulation - run and generate output GIF

In [ ]:
# =============================================================================
# CELL 1: SETUP - Imports, Classes, and Model Creation
# =============================================================================

import mujoco
import numpy as np
import os
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Image as IPImage, display
import tempfile

print("MuJoCo version:", mujoco.__version__)

# -----------------------------------------------------------------------------
# XML Creation Function
# -----------------------------------------------------------------------------

def create_simple_xml(num_fruits=5):
    """
    Create a MuJoCo XML with the UR10e robot and fruit balls.
    """
    xml_string = f"""
<mujoco model="robot_free_fall">
  <compiler angle="radian" meshdir="mujoco_menagerie/universal_robots_ur10e/assets" autolimits="true"/>
  
  <option integrator="implicitfast" timestep="0.002"/>
  
  <statistic center="0.4 0 0.4" extent="1.5"/>

  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0 0 0"/>
    <rgba haze="0.15 0.25 0.35 1"/>
    <global azimuth="120" elevation="-20" offwidth="1280" offheight="720"/>
  </visual>

  <default>
    <default class="ur10e">
      <material specular="0.5" shininess="0.25"/>
      <joint axis="0 1 0" range="-6.28319 6.28319" armature="0.1"/>
      <default class="size4">
        <joint damping="10"/>
      </default>
      <default class="size3">
        <joint damping="5"/>
        <default class="size3_limited">
          <joint range="-3.1415 3.1415"/>
        </default>
      </default>
      <default class="size2">
        <joint damping="2"/>
      </default>
      <default class="visual">
        <geom type="mesh" contype="0" conaffinity="0" group="2"/>
      </default>
      <default class="collision">
        <geom type="capsule" group="3"/>
        <default class="eef_collision">
          <geom type="cylinder"/>
        </default>
      </default>
      <site size="0.001" rgba="0.5 0.5 0.5 0.3" group="4"/>
    </default>
    <default class="fruit">
      <geom type="sphere" size="0.05" mass="0.1" friction="0.5 0.5 0.5"/>
    </default>
  </default>

  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.1 0.15 0.3" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1="0.15 0.2 0.25" rgb2="0.1 0.15 0.2"
      markrgb="0.3 0.3 0.3" width="300" height="300"/>
    <material name="groundplane" texture="groundplane" texuniform="true" texrepeat="5 5" reflectance="0.2"/>
    
    <!-- Robot meshes -->
    <material class="ur10e" name="black" rgba="0.033 0.033 0.033 1"/>
    <material class="ur10e" name="jointgray" rgba="0.278 0.278 0.278 1"/>
    <material class="ur10e" name="linkgray" rgba="0.82 0.82 0.82 1"/>
    <material class="ur10e" name="urblue" rgba="0.49 0.678 0.8 1"/>
    
    <mesh file="base_0.obj"/>
    <mesh file="base_1.obj"/>
    <mesh file="shoulder_0.obj"/>
    <mesh file="shoulder_1.obj"/>
    <mesh file="shoulder_2.obj"/>
    <mesh file="upperarm_0.obj"/>
    <mesh file="upperarm_1.obj"/>
    <mesh file="upperarm_2.obj"/>
    <mesh file="upperarm_3.obj"/>
    <mesh file="forearm_0.obj"/>
    <mesh file="forearm_1.obj"/>
    <mesh file="forearm_2.obj"/>
    <mesh file="forearm_3.obj"/>
    <mesh file="wrist1_0.obj"/>
    <mesh file="wrist1_1.obj"/>
    <mesh file="wrist1_2.obj"/>
    <mesh file="wrist2_0.obj"/>
    <mesh file="wrist2_1.obj"/>
    <mesh file="wrist2_2.obj"/>
    <mesh file="wrist3.obj"/>
  </asset>

  <worldbody>
    <light pos="0 0 3" dir="0 0 -1" directional="true" diffuse="0.8 0.8 0.8"/>
    <light pos="2 2 2" dir="-1 -1 -1" diffuse="0.3 0.3 0.3"/>
    <geom name="floor" size="0 0 0.05" type="plane" material="groundplane"/>
    
    <!-- UR10e Robot -->
    <body name="base" quat="0 0 0 -1" childclass="ur10e">
      <inertial mass="4.0" pos="0 0 0" diaginertia="0.0061063308908 0.0061063308908 0.01125"/>
      <geom mesh="base_0" material="black" class="visual"/>
      <geom mesh="base_1" material="jointgray" class="visual"/>
      <body name="shoulder_link" pos="0 0 0.181">
        <inertial pos="0 0 0" mass="7.778" diaginertia="0.0314743 0.0314743 0.0218756"/>
        <joint name="shoulder_pan_joint" class="size4" axis="0 0 1"/>
        <geom mesh="shoulder_0" material="urblue" class="visual"/>
        <geom mesh="shoulder_1" material="black" class="visual"/>
        <geom mesh="shoulder_2" material="jointgray" class="visual"/>
        <geom class="collision" size="0.078 0.08" pos="0 0 -0.05"/>
        <body name="upper_arm_link" pos="0 0.176 0" quat="1 0 1 0">
          <inertial pos="0 0 0.3065" mass="12.93" diaginertia="0.423074 0.423074 0.0363656"/>
          <joint name="shoulder_lift_joint" class="size4"/>
          <geom mesh="upperarm_0" material="black" class="visual"/>
          <geom mesh="upperarm_1" material="jointgray" class="visual"/>
          <geom mesh="upperarm_2" material="urblue" class="visual"/>
          <geom mesh="upperarm_3" material="linkgray" class="visual"/>
          <geom class="collision" pos="0 -0.05 0" quat="1 1 0 0" size="0.078 0.08"/>
          <geom class="collision" size="0.06 0.3" pos="0 0 0.3"/>
          <body name="forearm_link" pos="0 -0.137 0.613">
            <inertial pos="0 0 0.2855" mass="3.87" diaginertia="0.11059 0.11059 0.0108844"/>
            <joint name="elbow_joint" class="size3_limited"/>
            <geom mesh="forearm_0" material="urblue" class="visual"/>
            <geom mesh="forearm_1" material="black" class="visual"/>
            <geom mesh="forearm_2" material="jointgray" class="visual"/>
            <geom mesh="forearm_3" material="linkgray" class="visual"/>
            <geom class="collision" pos="0 0.08 0" quat="1 1 0 0" size="0.058 0.065"/>
            <geom class="collision" size="0.043 0.28" pos="0 0 0.29"/>
            <body name="wrist_1_link" pos="0 0 0.571" quat="1 0 1 0">
              <inertial pos="0 0.135 0" quat="0.5 0.5 -0.5 0.5" mass="1.96" diaginertia="0.0055125 0.00510825 0.00510825"/>
              <joint name="wrist_1_joint" class="size2"/>
              <geom mesh="wrist1_0" material="black" class="visual"/>
              <geom mesh="wrist1_1" material="urblue" class="visual"/>
              <geom mesh="wrist1_2" material="jointgray" class="visual"/>
              <geom class="collision" pos="0 0.06 0" quat="1 1 0 0" size="0.05 0.07"/>
              <body name="wrist_2_link" pos="0 0.135 0">
                <inertial pos="0 0 0.12" quat="0.5 0.5 -0.5 0.5" mass="1.96" diaginertia="0.0055125 0.00510825 0.00510825"/>
                <joint name="wrist_2_joint" axis="0 0 1" class="size2"/>
                <geom mesh="wrist2_0" material="black" class="visual"/>
                <geom mesh="wrist2_1" material="urblue" class="visual"/>
                <geom mesh="wrist2_2" material="jointgray" class="visual"/>
                <geom class="collision" size="0.046 0.065" pos="0 0 0.05"/>
                <geom class="collision" pos="0 0.028 0.12" quat="1 1 0 0" size="0.046 0.043"/>
                <body name="wrist_3_link" pos="0 0 0.12">
                  <inertial pos="0 0.092 0" quat="0 1 -1 0" mass="0.202" diaginertia="0.000204525 0.000144346 0.000144346"/>
                  <joint name="wrist_3_joint" class="size2"/>
                  <geom material="linkgray" mesh="wrist3" class="visual"/>
                  <geom class="eef_collision" pos="0 0.097 0" quat="1 1 0 0" size="0.046 0.02"/>
                  <site name="eef_site" pos="0 0.15 0" size="0.02" rgba="1 0 0 0.3"/>
                </body>
              </body>
            </body>
          </body>
        </body>
      </body>
    </body>
    
    <!-- Fruit balls -->
    {"".join([f'''
    <body name="fruit_{i}" pos="{2 + i*0.5} {-2 + (i%3)*0.3} {0.5 + i*0.2}">
      <freejoint name="fruit_{i}_joint"/>
      <geom name="fruit_{i}_geom" class="fruit" rgba="0.9 0.1 0.1 1"/>
    </body>''' for i in range(num_fruits)])}
    
  </worldbody>

  <actuator>
    <motor name="shoulder_pan" joint="shoulder_pan_joint" gear="1" ctrllimited="true" ctrlrange="-330 330"/>
    <motor name="shoulder_lift" joint="shoulder_lift_joint" gear="1" ctrllimited="true" ctrlrange="-330 330"/>
    <motor name="elbow" joint="elbow_joint" gear="1" ctrllimited="true" ctrlrange="-150 150"/>
    <motor name="wrist_1" joint="wrist_1_joint" gear="1" ctrllimited="true" ctrlrange="-56 56"/>
    <motor name="wrist_2" joint="wrist_2_joint" gear="1" ctrllimited="true" ctrlrange="-56 56"/>
    <motor name="wrist_3" joint="wrist_3_joint" gear="1" ctrllimited="true" ctrlrange="-56 56"/>
  </actuator>

  <keyframe>
    <key name="home" qpos="-1.5708 -1.5708 1.5708 -1.5708 -1.5708 0"/>
  </keyframe>
</mujoco>
"""
    return xml_string


# -----------------------------------------------------------------------------
# Fruit Manager Class
# -----------------------------------------------------------------------------

class FruitManager:
    """Manages fruit spawning and proximity detection"""
    
    def __init__(self, model, data, num_fruits):
        self.model = model
        self.data = data
        self.num_fruits = num_fruits
        
        # Get fruit body and geom IDs
        self.fruit_body_ids = []
        self.fruit_geom_ids = []
        self.fruit_qpos_addrs = []
        self.fruit_qvel_addrs = []
        
        for i in range(num_fruits):
            body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, f"fruit_{i}")
            geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, f"fruit_{i}_geom")
            joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, f"fruit_{i}_joint")
            
            self.fruit_body_ids.append(body_id)
            self.fruit_geom_ids.append(geom_id)
            self.fruit_qpos_addrs.append(model.jnt_qposadr[joint_id])
            self.fruit_qvel_addrs.append(model.jnt_dofadr[joint_id])
        
        # Fruit states: 0=waiting, 1=flying
        self.fruit_states = np.zeros(num_fruits, dtype=int)
        self.fruit_spawn_times = np.full(num_fruits, -1.0)
        
        # Scoring
        self.score = 0
        
    def spawn_fruit(self, fruit_idx, current_time):
        """Launch a fruit from outside the workspace toward the center"""
        if self.fruit_states[fruit_idx] != 0:
            return
        
        # Deterministic spawn positions based on fruit index
        spawn_angles = [0.0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi, 5*np.pi/4, 3*np.pi/2, 7*np.pi/4]
        spawn_heights = [0.3, 0.5, 0.7, 0.9, 0.4, 0.6, 0.8, 0.5]
        spawn_distances = [2.0, 2.5, 2.2, 1.8, 2.3, 2.0, 1.9, 2.4]
        
        spawn_angle = spawn_angles[fruit_idx % len(spawn_angles)]
        spawn_height = spawn_heights[fruit_idx % len(spawn_heights)]
        spawn_distance = spawn_distances[fruit_idx % len(spawn_distances)]
        
        spawn_pos = np.array([
            spawn_distance * np.cos(spawn_angle),
            spawn_distance * np.sin(spawn_angle),
            spawn_height
        ])
        
        # Deterministic target positions
        target_heights = [0.8, 1.2, 1.0, 0.9, 1.3, 0.7, 1.1, 0.95]
        target_x_offsets = [0.0, 0.3, -0.2, 0.4, -0.3, 0.2, -0.4, 0.1]
        target_y_offsets = [0.0, -0.2, 0.3, -0.3, 0.2, 0.4, -0.1, 0.3]
        
        target_pos = np.array([
            target_x_offsets[fruit_idx % len(target_x_offsets)],
            target_y_offsets[fruit_idx % len(target_y_offsets)],
            target_heights[fruit_idx % len(target_heights)]
        ])
        
        # Deterministic flight times
        flight_times = [1.0, 0.9, 1.1, 0.95, 1.05, 0.85, 1.15, 1.0]
        flight_time = flight_times[fruit_idx % len(flight_times)]
        gravity = 9.81
        
        velocity = (target_pos - spawn_pos) / flight_time
        velocity[2] += 0.5 * gravity * flight_time  # Account for gravity
        
        # Set position and velocity in MuJoCo
        qpos_addr = self.fruit_qpos_addrs[fruit_idx]
        qvel_addr = self.fruit_qvel_addrs[fruit_idx]
        
        # Position (x, y, z) + quaternion (w, x, y, z)
        self.data.qpos[qpos_addr:qpos_addr+3] = spawn_pos
        self.data.qpos[qpos_addr+3:qpos_addr+7] = [1, 0, 0, 0]  # Identity quaternion
        
        # Velocity (linear + angular)
        self.data.qvel[qvel_addr:qvel_addr+3] = velocity
        self.data.qvel[qvel_addr+3:qvel_addr+6] = [0, 0, 0]  # No rotation
        
        self.fruit_states[fruit_idx] = 1  # Flying
        self.fruit_spawn_times[fruit_idx] = current_time
        
        print(f"  Spawned fruit {fruit_idx} at ({spawn_pos[0]:.2f}, {spawn_pos[1]:.2f}, {spawn_pos[2]:.2f}) with velocity ({velocity[0]:.2f}, {velocity[1]:.2f}, {velocity[2]:.2f})")
        
    def get_fruit_position(self, fruit_idx):
        """Get current position of a fruit"""
        body_id = self.fruit_body_ids[fruit_idx]
        return self.data.xpos[body_id].copy()
    
    def reset_fruit(self, fruit_idx):
        """Reset a fruit to waiting state"""
        self.fruit_states[fruit_idx] = 0
        # Move far away
        qpos_addr = self.fruit_qpos_addrs[fruit_idx]
        self.data.qpos[qpos_addr:qpos_addr+3] = [10, 10, -10]
        self.data.qvel[self.fruit_qvel_addrs[fruit_idx]:self.fruit_qvel_addrs[fruit_idx]+6] = 0


# -----------------------------------------------------------------------------
# Robot Controller Class
# -----------------------------------------------------------------------------

class RobotController:
    """Handles robot joint torque control with time-varying torques"""

    def __init__(self, model, data, joint_torques_matrix):
        self.model = model
        self.data = data
        self.num_joints = 6  # UR10e has 6 joints
        self.joint_names = [
            "shoulder_pan_joint",
            "shoulder_lift_joint",
            "elbow_joint",
            "wrist_1_joint",
            "wrist_2_joint",
            "wrist_3_joint"
        ]

        # Convert to numpy array if needed
        self.torques_matrix = np.array(joint_torques_matrix, dtype=float)

        # Validate shape
        if self.torques_matrix.ndim == 1:
            # Single row provided, reshape to (1, 6)
            self.torques_matrix = self.torques_matrix.reshape(1, -1)

        if self.torques_matrix.shape[1] != self.num_joints:
            raise ValueError(f"Expected {self.num_joints} joints (columns), got {self.torques_matrix.shape[1]}")

        self.num_steps = self.torques_matrix.shape[0]
        self.current_torques = np.zeros(self.num_joints)

    def get_torques_at_step(self, step):
        """Get torques for a specific time step"""
        if step < self.num_steps:
            return self.torques_matrix[step]
        else:
            # If simulation runs longer than specified torques, use the last row
            return self.torques_matrix[-1]

    def apply_torques(self, data, step):
        """Apply the torque values to the robot for a given time step"""
        self.current_torques = self.get_torques_at_step(step)
        data.ctrl[:self.num_joints] = self.current_torques

    def has_any_control(self):
        """Check if any torques are non-zero"""
        return np.any(self.torques_matrix != 0)

    def print_torques_info(self):
        """Print information about the torque configuration"""
        print(f"\nTorque Matrix Shape: {self.torques_matrix.shape[0]} steps x {self.torques_matrix.shape[1]} joints")
        if self.has_any_control():
            print("Control Status: ACTIVE")
            print("\nTorque Statistics (N·m):")
            for i, name in enumerate(self.joint_names):
                min_val = np.min(self.torques_matrix[:, i])
                max_val = np.max(self.torques_matrix[:, i])
                mean_val = np.mean(self.torques_matrix[:, i])
                print(f"  {name}: min={min_val:>7.2f}, max={max_val:>7.2f}, mean={mean_val:>7.2f}")
        else:
            print("Control Status: NO CONTROL (all torques are zero)")


# -----------------------------------------------------------------------------
# Helper Functions for Creating Torque Patterns
# -----------------------------------------------------------------------------

def constant_torques(torques, num_steps):
    """Create constant torques for all time steps

    Args:
        torques: List of 6 torque values
        num_steps: Number of time steps
    Returns:
        Matrix of shape (num_steps, 6)
    """
    return np.tile(torques, (num_steps, 1))

def ramp_torques(start_torques, end_torques, num_steps):
    """Create linearly ramping torques

    Args:
        start_torques: List of 6 initial torque values
        end_torques: List of 6 final torque values
        num_steps: Number of time steps
    Returns:
        Matrix of shape (num_steps, 6)
    """
    return np.linspace(start_torques, end_torques, num_steps)

def save_torques(torques, filename):
    """Save torque matrix to file"""
    np.save(filename, torques)
    print(f"Torques saved to {filename}")

def load_torques(filename):
    """Load torque matrix from file"""
    return np.load(filename)


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

NUM_FRUITS = 8

# Create and save the XML
xml_content = create_simple_xml(NUM_FRUITS)
xml_path = "robot_free_fall.xml"
with open(xml_path, 'w') as f:
    f.write(xml_content)

# Load the model
model = mujoco.MjModel.from_xml_string(xml_content)
data = mujoco.MjData(model)

# Get end effector site ID
eef_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "eef_site")

# Simulation duration
DURATION = 15.0  # seconds
TIMESTEP = 0.002  # seconds (from model)
NUM_STEPS = int(DURATION / TIMESTEP)  # 7500 steps for 15 seconds


In [ ]:
################### EDIT HERE ###########################
# NUM_STEPS is 7500
# each time step is 0.002s
JOINT_TORQUES = np.zeros((NUM_STEPS, 6))





#########################################################


In [ ]:
# =============================================================================
# CELL 3: SIMULATION AND OUTPUT
# =============================================================================

# Reset simulation
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

# Set robot to home position (upright)
home_qpos = np.array([-1.5708, -1.5708, 1.5708, -1.5708, -1.5708, 0])
data.qpos[:6] = home_qpos

# Initialize fruit manager
fruit_manager = FruitManager(model, data, NUM_FRUITS)

# Track which fruits have been touched (turned green)
touched_fruits = set()

# Initialize robot controller with specified torques
robot_controller = RobotController(model, data, JOINT_TORQUES)

# Camera setup
camera = mujoco.MjvCamera()
camera.distance = 4.0
camera.azimuth = 135
camera.elevation = -25
camera.lookat[:] = [0.0, 0.0, 0.6]

# Create renderer
renderer = mujoco.Renderer(model, height=720, width=1280)

# Simulation parameters
duration = DURATION
fps = 10
timestep = model.opt.timestep
frame_skip = int(1 / (timestep * fps))

# Spawning parameters
spawn_interval = 2.5  # seconds between spawns
last_spawn_time = -spawn_interval
fruits_to_spawn = list(range(NUM_FRUITS))
spawn_idx = 0

frames = []
total_steps = int(duration / timestep)

print(f"\nStarting simulation...")
print(f"Duration: {duration}s, FPS: {fps}, Timestep: {timestep}s")
print(f"Total steps: {total_steps}, Frame skip: {frame_skip}")
print("-" * 50)

for step in range(total_steps):
    current_time = data.time
    
    # Spawn new fruits periodically
    if current_time - last_spawn_time >= spawn_interval and spawn_idx < len(fruits_to_spawn):
        fruit_manager.spawn_fruit(fruits_to_spawn[spawn_idx], current_time)
        spawn_idx += 1
        last_spawn_time = current_time
    
    # Respawn fruits that fell below the floor
    for i in range(NUM_FRUITS):
        if fruit_manager.fruit_states[i] == 1:  # Flying
            fruit_pos = fruit_manager.get_fruit_position(i)
            if fruit_pos[2] < -0.5:  # Below floor
                spawn_time = fruit_manager.fruit_spawn_times[i]
                if current_time - spawn_time > 3.0:  # Wait 3 seconds before respawn
                    fruit_manager.reset_fruit(i)
                    fruits_to_spawn.append(i)
                    # Reset color to red when respawning
                    if i in touched_fruits:
                        touched_fruits.remove(i)
                        geom_id = fruit_manager.fruit_geom_ids[i]
                        model.geom_rgba[geom_id] = [0.9, 0.1, 0.1, 1.0]
    
    # Apply robot control torques (if any are non-zero)
    if robot_controller.has_any_control():
        robot_controller.apply_torques(data, step)

    # Check for proximity with end effector
    eef_pos = data.site_xpos[eef_site_id].copy()
    contact_radius = 0.15  # Distance threshold for contact detection
    
    for i in range(NUM_FRUITS):
        if fruit_manager.fruit_states[i] == 1 and i not in touched_fruits:  # Flying and not yet touched
            fruit_pos = fruit_manager.get_fruit_position(i)
            distance = np.linalg.norm(fruit_pos - eef_pos)
            
            if distance < contact_radius:
                # Change fruit color to green
                geom_id = fruit_manager.fruit_geom_ids[i]
                model.geom_rgba[geom_id] = [0.2, 0.8, 0.2, 1.0]
                touched_fruits.add(i)
                fruit_manager.score += 10
                print(f"[{current_time:.1f}s] ✓ Touched fruit {i}! Score: {fruit_manager.score}")
    
    # Step simulation
    mujoco.mj_step(model, data)
    
    # Render frames
    if step % frame_skip == 0:
        renderer.update_scene(data, camera=camera)
        pixels = renderer.render()
        frames.append(Image.fromarray(pixels.copy()))
        
        if len(frames) % 60 == 0:
            print(f"  Rendered {len(frames)} frames, Time: {current_time:.1f}s, Score: {fruit_manager.score}")

print("-" * 50)
print(f"Simulation complete!")
print(f"Final Score: {fruit_manager.score}")
print(f"Fruits touched: {len(touched_fruits)}")
print(f"Total frames: {len(frames)}")

renderer.close()

output_path = "robot_dynamics_simulation.gif"

# Save as GIF
frames[0].save(
    output_path,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000/fps),
    loop=0,
    optimize=True
)

file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"\nGIF saved to: {output_path} ({file_size:.2f} MB)")

# Display the GIF

display(IPImage(filename=output_path))